In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

class WearResidual(Model):
    def __init__(self, hidden_units=32):
        super().__init__()
        self.torque_net = tf.keras.Sequential([
            layers.Dense(hidden_units, activation="tanh"),
            layers.Dense(hidden_units, activation="tanh"),
            layers.Dense(1)
        ])
        self.wear_net = tf.keras.Sequential([
            layers.Dense(hidden_units, activation="tanh"),
            layers.Dense(1)  # wear rate
        ])

    def torque(self, omega, iq, fz, ap, ae, W):
        """
        omega, iq, fz, ap, ae, W : (batch, 1, 1)
        returns tau_cut: (batch, 1, 1)
        """
        x = tf.concat([omega, iq, fz, ap, ae, W], axis=-1)
        return self.torque_net(x)

    def wear_rate(self, fz, ap, ae, omega):
        """
        Returns non-negative wear rate dW/dt
        """
        x = tf.concat([fz, ap, ae, tf.abs(omega)], axis=-1)
        r = self.wear_net(x)
        # enforce non-negative wear rate
        return tf.nn.softplus(r)


class SpindleWearGrayBox(Model):
    def __init__(self, hidden_units=32, dt=0.001):
        super().__init__()
        self.dt = dt
        self.residual = WearResidual(hidden_units)

        def init_param(x0):
            return tf.Variable(
                tf.math.log(tf.exp(tf.constant(x0, tf.float32)) - 1.0),
                trainable=True
            )
        self.log_J  = init_param(0.01)
        self.log_B  = init_param(0.05)
        self.log_Kt = init_param(1.0)

        # initial wear level (can also be passed per-window)
        self.log_W0 = init_param(0.01)  # small positive

    def call(self, inputs):
        """
        inputs:
          iq:    (batch, T, 1)
          fz:    (batch, T, 1)
          ap:    (batch, T, 1)
          ae:    (batch, T, 1)
          omega0: (batch, 1)   measured initial speed
          # optionally: W0 per-window instead of global
        returns:
          omega_pred: (batch, T, 1)
          W_traj:     (batch, T, 1)
        """
        iq_seq   = inputs["iq"]
        fz_seq   = inputs["fz"]
        ap_seq   = inputs["ap"]
        ae_seq   = inputs["ae"]
        omega0   = inputs["omega0"]  # (batch, 1, 1) or (batch, 1)

        if len(omega0.shape) == 2:
            omega_t = omega0  # (batch, 1)
            omega_t = tf.expand_dims(omega_t, axis=-1)  # (batch, 1, 1)
        else:
            omega_t = omega0

        J  = tf.nn.softplus(self.log_J)
        B  = tf.nn.softplus(self.log_B)
        Kt = tf.nn.softplus(self.log_Kt)
        dt = self.dt

        # initial wear
        W0 = tf.nn.softplus(self.log_W0)  # scalar
        # broadcast across batch
        batch_size = tf.shape(iq_seq)[0]
        W_t = W0 * tf.ones((batch_size, 1, 1), dtype=tf.float32)

        T = tf.shape(iq_seq)[1]

        omega_pred = []
        W_traj = []

        for k in range(T):
            iq_k = iq_seq[:, k:k+1, :]
            fz_k = fz_seq[:, k:k+1, :]
            ap_k = ap_seq[:, k:k+1, :]
            ae_k = ae_seq[:, k:k+1, :]

            # cutting torque influenced by wear state
            tau_cut = self.residual.torque(omega_t, iq_k, fz_k, ap_k, ae_k, W_t)

            domega = (Kt*iq_k - B*omega_t - tau_cut) / J
            omega_t = omega_t + dt*domega

            dW = self.residual.wear_rate(fz_k, ap_k, ae_k, omega_t)
            W_t = W_t + dt*dW

            omega_pred.append(omega_t)
            W_traj.append(W_t)

        omega_pred = tf.concat(omega_pred, axis=1)
        W_traj     = tf.concat(W_traj, axis=1)
        return omega_pred, W_traj
